In [0]:
%pip install pyyaml 

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml

In [0]:
# orders_df = spark.read.parquet("/Volumes/pei/bronze/orders/")
# customers_df = spark.read.parquet("/Volumes/pei/bronze/customer/")
products_df = spark.read.parquet("/Volumes/pei/bronze/product/")

In [0]:
products_df.show(2)

In [0]:
# config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/silver/products.yaml"

# with open(config_path, "r") as f:
#     config = yaml.safe_load(f)

In [0]:
def read_source(source_config):
    return (
        spark.read
        .format(source_config["format"])
        .load(source_config["path"])
    )

In [0]:
def apply_schema_and_rename(df, schema_config):
    for col_def in schema_config:
        source_col = col_def["source_name"]
        target_col = col_def["name"]
        dtype = col_def["type"]

        if source_col in df.columns:
            df = df.withColumnRenamed(source_col, target_col)
            df = df.withColumn(target_col, F.col(target_col).cast(dtype))
        else:
            raise Exception(f"Column {source_col} not found in source")

    return df

In [0]:
def apply_transformations(df, transformations):
    for t in transformations:
        df = df.selectExpr("*", t["expr"])
    return df

In [0]:
def apply_deduplication(df, dedup_config):
    if not dedup_config:
        return df

    keys = dedup_config["keys"]
    order_col = dedup_config["order_by"]

    window = Window.partitionBy(*keys).orderBy(F.col(order_col).desc())

    df = (
        df.withColumn("rn", F.row_number().over(window))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

    return df

In [0]:
def select_columns(df, columns):
    return df.select(*columns)

In [0]:
def write_target(df, target_config):
    writer = (
        df.write
        .format(target_config["format"])
        .mode(target_config["mode"])
    )

    # Optional partitioning (not used for products)
    if "partition_by" in target_config:
        writer = writer.partitionBy(*target_config["partition_by"])

    # Write to path
    if "path" in target_config:
        writer.save(target_config["path"])

    # Register table
    if "table" in target_config:
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {target_config['table']}
            USING DELTA
            LOCATION '{target_config['path']}'
        """)

In [0]:
def apply_quality_checks(df, rules):
    for rule in rules:
        col = rule["column"]
        condition = rule["rule"]

        if condition == "not_null":
            df = df.filter(F.col(col).isNotNull())
        else:
            df = df.filter(f"{col} {condition}")

    return df

In [0]:
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
# Main fn
config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/silver/products.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f) 

def process_table(config):
    # config = load_yaml(config_path)
    print("started main")
    df = read_source(config["source"])
    print("completed read source")
    df = apply_schema_and_rename(df, config["schema"])
    df = apply_transformations(df, config.get("transformations", []))
    df = apply_deduplication(df, config.get("deduplication"))
    #df = select_columns(df, config["columns"]["select"])
    # write_target(df, config["target"])
    display(df)

process_table(config)

In [0]:
# def ingest_dataset(config):

#     reader_fn = get_reader(config["format"])

#     df = reader_fn(spark, config)

#     df = standardize(df, config)

#     write_to_bronze(df, config)

In [0]:
# for dataset in config["datasets"]:
#     try:
#         ingest_dataset(dataset)
#     except Exception as e:
#         print(f"Failed: {dataset['name']} → {str(e)}")

In [0]:
# for table_name in config["table_name"]:
#     try:
#         process_table(table_name)
#     except Exception as e:
#         print(f"Failed: {table_name} → {str(e)}")